<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "404"
iso_country = "Kenya"
year = 2020
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 100.660*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Kenya': 

Number of trade records present in dataset: 32895
Number of partner countries present in dataset: 151
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$12510.8m
Total weight within the dataset: 13178.9 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,404,Kenya,699,India,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,183953.222,NaN,183953.222,kg,32898.0,32898.0,28347.191510,4305.0,2675.002155,75828.798378,['Container'],['Container'],257.572910,256.624104,256.694835,256.694835,257.294932,257.294932,270.641375,509.269469,298.495209,275.500922,276.563552,284.470778,264.437671,484.450716,311.402575,286.553171,279.943997,294.074767,259.001720,537.003253,391.019959,340.430261,314.539330,307.334343,253.642223,569.658752,457.007089,426.391727,333.566018,330.569120,248.381239,581.580095,548.994422,558.283666,360.779346,347.728191,243.019194,571.261722,546.663364,580.468874,436.574899,354.336567,237.971708,505.714846,521.563876,545.687883,404.674383,358.668978,232.841733,455.691395,466.078941,538.860616,445.407651,361.486269,2676
1,1,404,Kenya,826,United Kingdom,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,6181.099,NaN,6181.099,kg,1775.0,1775.0,952.507356,11940.0,7419.169740,7066.813752,['Container'],['Container'],24.004334,23.915910,23.922502,23.922502,23.978428,23.978428,25.222241,47.461025,27.818060,25.675123,25.774154,26.511062,24.644090,45.148058,29.020953,26.705130,26.089192,27.406100,24.137491,50.045656,36.440841,31.726169,29.313281,28.641817,23.638016,53.088963,42.590468,39.737290,31.086460,30.807166,23.147722,54.199965,51.163165,52.028870,33.622588,32.406294,22.648010,53.238351,50.945924,54.096405,40.686303,33.022158,22.177613,47.129754,48.606794,50.855014,37.713356,33.425914,21.699528,42.467852,43.435913,50.218752,41.509466,33.688469,2676


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $2132.5m (17.0%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
15 | Animal, vegetable or  791701606.8  9.186529e+08      6.3      7.0
10 | Cereals               673941503.9  2.068631e+09      5.4     15.7
17 | Sugars and sugar con  235011549.5  4.951193e+08      1.9      3.8
21 | Miscellaneous edible   88964370.8  2.202646e+07      0.7      0.2
22 | Beverages, spirits a   79579555.8  4.173317e+07      0.6      0.3
19 | Preparations of cere   52957589.1  2.367142e+07      0.4      0.2
07 | Vegetables and certa   31471573.1  4.726450e+07      0.3      0.4
20 | Preparations of vege   29202153.6  1.599493e+07      0.2      0.1
08 | Fruit and nuts, edib   28328304.1  2.518505e+07      0.2      0.2
23 | Food industries, res   23733487.1  1.988418e+07      0.2      0.2
12 | Oil seeds and oleagi   20368096.6  2.5468

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $1550.1m (12.4%), ... 

                      cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                     
Agriculture       1.148316e+09  1.548795e+09      9.2     11.8
Food & Beverages  4.016171e+08  3.475911e+08      3.2      2.6
Fishing           1.573082e+05  6.829060e+04      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $1480.2m (11.8%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
VOL - Vegetable oils and fats                 791736453.3  919643405.0   
OFD - Food products n.e.c.                    244916225.3  128247501.0   
PCR - Processed rice                          208523366.9  362205088.3   
SGR - Sugar                                    94104096.6  206455545.7   
V_F - Vegetables, fruit, nuts                  57763836.9   70767738.8   
GRO - Cereal grains n.e.c.                     38174668.3  157877928.0   
MIL - Dairy products                           21566689.4    7831698.7   
OCR - Crops n.e.c.                             17632451.8    3692157.0   
OMT - Meat products n.e.c.                      3570320.3    1331480.6   
OSD - Oil seeds                                 1003776.1    1093138.3   
OAP - Animal products n.e.c.                 

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 17.0% 
GTAP 11.8% 
EORA 12.4%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46385/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,404,Kenya,699,India,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,183953.222,NaN,183953.222,kg,32898.0,32898.0,28347.191510,4305.0,2675.002155,75828.798378,['Container'],['Container'],257.572910,256.624104,256.694835,256.694835,257.294932,257.294932,270.641375,509.269469,298.495209,275.500922,276.563552,284.470778,264.437671,484.450716,311.402575,286.553171,279.943997,294.074767,259.001720,537.003253,391.019959,340.430261,314.539330,307.334343,253.642223,569.658752,457.007089,426.391727,333.566018,330.569120,248.381239,581.580095,548.994422,558.283666,360.779346,347.728191,243.019194,571.261722,546.663364,580.468874,436.574899,354.336567,237.971708,505.714846,521.563876,545.687883,404.674383,358.668978,232.841733,455.691395,466.078941,538.860616,445.407651,361.486269,2676,NaN
1,1,404,Kenya,826,United Kingdom,Import,Sea,551321,"551321 | Fabrics, woven; plai",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,6181.099,NaN,6181.099,kg,1775.0,1775.0,952.507356,11940.0,7419.169740,7066.813752,['Container'],['Container'],24.004334,23.915910,23.922502,23.922502,23.978428,23.978428,25.222241,47.461025,27.818060,25.675123,25.774154,26.511062,24.644090,45.148058,29.020953,26.705130,26.089192,27.406100,24.137491,50.045656,36.440841,31.726169,29.313281,28.641817,23.638016,53.088963,42.590468,39.737290,31.086460,30.807166,23.147722,54.199965,51.163165,52.028870,33.622588,32.406294,22.648010,53.238351,50.945924,54.096405,40.686303,33.022158,22.177613,47.129754,48.606794,50.855014,37.713356,33.425914,21.699528,42.467852,43.435913,50.218752,41.509466,33.688469,2676,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $1171.4m of $12510.8m (9.4%)
Volume: 2734.9 of 13178.9 million kilograms (20.8%)
Value-to-Weight: 0.43 from 0.95 US$ per kg (+-54.9%)
Average distance to market (volume-weighted): 5637.7 from 4702.7 nm (+19.9%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     62  465360840.3  1.706326e+09      3.7     12.9
1    Processed Cereals    683  348148597.0  4.240494e+08      2.8      3.2
2      Sugar and Honey    159  235036360.4  4.951265e+08      1.9      3.8
3           Vegetables    252   40419908.2  5.108477e+07      0.3      0.4
4        Fats and Oils    138   32890923.4  2.524231e+07      0.3      0.2
5               Fruits    180   28228733.0  2.514008e+07      0.2      0.2
6                Dairy     83   18238319.6  6.637109e+06      0.1      0.1
7        Meat and Prep     20    3083922.6  1.246523e+06      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.767017e+07,3.323560e+07,1.948660e+07,1.798636e+07,1.805569e+07,1.857232e+07,1.621803e+07,3.795284e+07,3.583005e+07,3.643878e+07,2.355179e+07,2.269897e+07,1.520432e+07,2.974287e+07,3.041995e+07,3.516949e+07,2.907160e+07,2.359644e+07
1,Fats and Oils,5.803824e+07,1.016098e+08,6.289343e+07,5.851637e+07,5.871410e+07,6.060403e+07,5.390728e+07,1.148604e+08,1.104557e+08,1.136487e+08,7.555986e+07,7.237024e+07,5.102367e+07,9.291087e+07,9.452709e+07,1.089577e+08,9.068958e+07,7.490446e+07
2,Fruits,2.457420e+07,4.603221e+07,2.706662e+07,2.499877e+07,2.509632e+07,2.581779e+07,2.257023e+07,5.262666e+07,4.966435e+07,5.055496e+07,3.271361e+07,3.151047e+07,2.117196e+07,4.127040e+07,4.220808e+07,4.878430e+07,4.034391e+07,3.275926e+07
3,Meat and Prep,2.711456e+06,5.102184e+06,2.990514e+06,2.760143e+06,2.770789e+06,2.850008e+06,2.488440e+06,5.826637e+06,5.500174e+06,5.593239e+06,3.614516e+06,3.483761e+06,2.332756e+06,4.565405e+06,4.669474e+06,5.398647e+06,4.462376e+06,3.621598e+06
4,Processed Cereals,5.350012e+08,1.000892e+09,5.890544e+08,5.441561e+08,5.462758e+08,5.620054e+08,4.914823e+08,1.144639e+09,1.080322e+09,1.099811e+09,7.121928e+08,6.856168e+08,4.611167e+08,8.979385e+08,9.182911e+08,1.061208e+09,8.777732e+08,7.127124e+08
5,Sugar and Honey,3.583982e+08,6.338720e+08,3.885813e+08,3.619750e+08,3.632787e+08,3.744754e+08,3.323975e+08,7.351469e+08,6.971530e+08,7.132004e+08,4.765808e+08,4.479683e+08,3.142480e+08,5.854149e+08,5.972730e+08,6.857560e+08,5.720754e+08,4.635135e+08
6,Unprocessed Cereals,4.287439e+09,7.973820e+09,4.713169e+09,4.357807e+09,4.374396e+09,4.501340e+09,3.942860e+09,9.132104e+09,8.627961e+09,8.783711e+09,5.712801e+09,5.479628e+09,3.702315e+09,7.177229e+09,7.337369e+09,8.472046e+09,7.014909e+09,5.691337e+09
7,Vegetables,1.271789e+08,2.381230e+08,1.400621e+08,1.293730e+08,1.298752e+08,1.336100e+08,1.168191e+08,2.722715e+08,2.569845e+08,2.615630e+08,1.693317e+08,1.630504e+08,1.095848e+08,2.135403e+08,2.183916e+08,2.524130e+08,2.087383e+08,1.694889e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.85,0.10,0.02,0.02,0.05,1.19,1.08,1.11,0.40,0.36,0.80,0.83,1.09,0.76,0.46
1,Fats and Oils,1.32,0.15,0.01,0.02,0.08,1.85,1.72,1.82,0.66,0.56,1.27,1.32,1.76,1.21,0.73
2,Fruits,0.76,0.09,0.02,0.02,0.04,1.06,0.96,0.99,0.36,0.32,0.71,0.75,0.98,0.68,0.41
3,Meat and Prep,0.78,0.09,0.02,0.02,0.04,1.08,0.98,1.01,0.37,0.32,0.72,0.76,0.99,0.69,0.42
4,Processed Cereals,1.34,0.16,0.03,0.03,0.08,1.88,1.69,1.75,0.63,0.56,1.25,1.31,1.72,1.20,0.72
5,Sugar and Honey,1.17,0.13,0.02,0.02,0.07,1.71,1.55,1.62,0.61,0.49,1.15,1.20,1.58,1.10,0.64
6,Unprocessed Cereals,7.92,0.91,0.15,0.19,0.46,11.15,10.07,10.40,3.80,3.30,7.47,7.81,10.25,7.12,4.27
7,Vegetables,2.74,0.32,0.05,0.07,0.16,3.85,3.47,3.58,1.30,1.14,2.57,2.69,3.53,2.45,1.48
